<a href="https://colab.research.google.com/github/Jozen1/EntregasGO_Equipo8/blob/main/notebooks/Avance1A.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Clona el repositorio 'Operations-Management' de GitHub en el entorno actual.
!git clone https://github.com/LuisRojo-Gonzalez/Operations-Management.git

Cloning into 'Operations-Management'...
remote: Enumerating objects: 445, done.
remote: Counting objects: 100% (445/445), done.
remote: Compressing objects: 100% (437/437), done.
remote: Total 445 (delta 32), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (445/445), 15.49 MiB | 12.45 MiB/s, done.
Resolving deltas: 100% (32/32), done.


In [ ]:
# Muestra la ruta del directorio de trabajo actual.
!pwd

/content


In [ ]:
# Importa las librerías necesarias
import pandas as pd
import numpy as np

In [ ]:
# Carga de datos
d = pd.read_csv('/content/Operations-Management/CaseStudy/Datos/calidad_llenado.csv')
specs = pd.read_csv('/content/Operations-Management/CaseStudy/Datos/especificaciones_calidad.csv')

In [ ]:
# Definición de la Fase (Ej: semanas 1 a 8 son Fase I, el resto Fase II)
d['fase'] = np.where(d['semana_registro'] <= 8, 'I', 'II')

In [1]:
# Constantes para n=5 (tamaño del subgrupo indicado en el caso)
A2 = 0.577
d2 = 2.326
D3 = 0        # Límite Inferior para la carta R
D4 = 2.114    # Límite Superior para la carta R

In [ ]:
# Lista para guardar el resumen de todas las líneas y formatos
resumen_final = []

In [ ]:
# Agrupamos el dataframe original para obtener cada combinación existente
combinaciones = d.groupby(['linea', 'formato_L'])

In [ ]:
for (linea, formato), df_analisis in combinaciones:

    # Obtener especificaciones exactas para este formato
    # Asumimos que las columnas se llaman 'formato_L', 'LIE', 'LSE' y 'contenido_nominal'
    spec_formato = specs[specs['formato_L'] == formato].iloc[0]
    LIE = spec_formato['LIE_mL']
    LSE = spec_formato['LSE_mL']
    nominal = spec_formato['nominal_mL']

    # Agrupar por fase y subgrupo para esta combinación específica
    g = df_analisis.groupby(['fase', 'subgrupo'])['volumen_mL']
    res = pd.DataFrame({'xbar': g.mean(), 'R': g.max() - g.min()}).reset_index()

    # Fase I: SOLO para calibrar los límites
    f1 = res[res['fase'] == 'I']

    # Si por algún motivo no hay datos de Fase I para esta línea, la saltamos
    if f1.empty:
        continue

    xbb = f1['xbar'].mean()
    Rb = f1['R'].mean()

    # Calcular Límites Congelados
    lcs_x = xbb + A2 * Rb
    lci_x = xbb - A2 * Rb

    # Evaluar toda la serie (Fase I y II) buscando puntos fuera de control
    res['fuera'] = (res['xbar'] > lcs_x) | (res['xbar'] < lci_x)
    total_alertas = res['fuera'].sum()

    # Análisis de Capacidad (Exclusivamente con la variabilidad de la Fase I)
    sigma_c = Rb / d2
    Cp = (LSE - LIE) / (6 * sigma_c)
    Cpk = min(LSE - xbb, xbb - LIE) / (3 * sigma_c)

    # Calcular el "Sobrellenado" medio (Clave para la planificación)
    sobrellenado_mL = xbb - nominal

    # Guardar resultados de esta iteración
    resumen_final.append({
        'Linea': linea,
        'Formato (L)': formato,
        'X_bar (Media)': round(xbb, 3),
        'Cp': round(Cp, 3),
        'Cpk': round(Cpk, 3),
        'Subgrupos c/Alarma': total_alertas,
        'Sobrellenado (mL)': round(sobrellenado_mL, 3)
    })

In [ ]:
# MOSTRAR RESULTADOS
df_resumen = pd.DataFrame(resumen_final)
print("=== Resumen de Control y Capacidad por Línea y Formato ===")
print(df_resumen.to_string(index=False))

=== Resumen de Control y Capacidad por Línea y Formato ===
Linea  Formato (L)  X_bar (Media)    Cp   Cpk  Subgrupos c/Alarma  Sobrellenado (mL)
   L1          0.5        503.708 1.281 0.687                  20              3.708
   L2          1.5       1504.660 0.977 0.673                   5              4.660
   L2          3.0       3008.550 0.967 0.636                   2              8.550
